In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import DataFrame



In [0]:
def merge_delta_produto(df, tabela_destino):
    """
    Realiza MERGE (upsert) de um DataFrame em uma tabela Delta Lake de configurações.

    Regras de chave:
    ----------------
    - Chave de negócio: (cliente, produto)
    - Mantém sempre APENAS 1 linha por (cliente, produto) na tabela destino.
    - Caso o DataFrame contenha mais de um registro para o mesmo (cliente, produto),
      será mantida a linha com a data_execucao mais recente.

    Parâmetros:
    -----------
    df : DataFrame
        DataFrame com os dados a serem inseridos/atualizados.
        Deve conter, obrigatoriamente:
            - cliente
            - produto
            - data_execucao
        Os demais campos podem ser nulos.
        
    tabela_destino : str
        Nome da tabela Delta Lake (ex: "workspace.bronze_etl.config_clientes").

    Retorno:
    --------
    None
    """

    # ⚠️ Verifica se há dados
    if df is None or df.limit(1).count() == 0:
        print("⚠️ Nenhum dado para atualizar.")
        return

    # 🔁 Deduplica por (cliente, produto) mantendo somente a última execução
    janela = Window.partitionBy("cliente", "produto").orderBy(F.col("data_execucao").desc())

    df_dedup = (
        df
        .withColumn("_rn", F.row_number().over(janela))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    total = df_dedup.count()
    print(f"✅ Total de registros (deduplicados) a carregar: {total}")

    # 🔍 Verifica se a tabela Delta já existe
    if spark.catalog.tableExists(tabela_destino):
        print(f"📦 Tabela {tabela_destino} já existe — atualizando dados...")

        delta_table = DeltaTable.forName(spark, tabela_destino)

        # 🔑 Condição de MERGE pela chave de negócio
        condicao_merge = "t.cliente = s.cliente AND t.produto = s.produto"
        print("🔑 Usando chaves: cliente, produto")

        (
            delta_table.alias("t")
            .merge(
                df_dedup.alias("s"),
                condicao_merge
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"✅ MERGE concluído com sucesso em {tabela_destino}")

    else:
        print(f"🆕 Tabela {tabela_destino} não existe — criando nova tabela...")

        (
            df_dedup.write
            .format("delta")
            .partitionBy("cliente")  # opcional
            .mode("append")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_destino)
        )

        print(f"✅ Tabela {tabela_destino} criada e dados inseridos com sucesso.")

In [0]:
def merge_config(df, tabela_destino: str):
    """
    Realiza MERGE da tabela de configuração de clientes (contabilidade)
    usando como chaves: cliente + mes_referencia.

    Parâmetros:
    - df_config: DataFrame Spark com as colunas padronizadas
    - tabela_destino: tabela Delta no formato catalogo.schema.tabela
    """

    # 🚨 Validação mínima
    required_cols = {"cliente", "mes_referencia"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"❌ O DataFrame precisa conter as colunas: {required_cols}. "
            f"Colunas recebidas: {df.columns}"
        )

    if spark.catalog.tableExists(tabela_destino):
        print(f"🔄 Tabela {tabela_destino} encontrada — realizando MERGE...")

        delta_tbl = DeltaTable.forName(spark, tabela_destino)

        (
            delta_tbl.alias("t")
            .merge(
                df.alias("s"),
                """
                t.cliente = s.cliente
                AND t.mes_referencia = s.mes_referencia
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("✅ MERGE concluído com sucesso.")
    else:
        print(f"🆕 Criando tabela {tabela_destino} pela primeira vez...")

        df.write \
                .format("delta") \
                .partitionBy("mes_referencia") \
                .mode("append") \
                .option("overwriteSchema", "true") \
                .saveAsTable(tabela_destino)

        print(" Tabela criada com sucesso.")

In [0]:
def calcular_custos_e_margens_silver(
    df_produtos: DataFrame,
    df_config: DataFrame
) -> DataFrame:
    """
    Gera a camada Silver de custos e margens por cliente/produto/mês.

    Grão do resultado:
        - 1 linha por (cliente, produto, mes_referencia)

    Entradas:
    ---------
    df_produtos:
        Schema esperado:
            - cliente (string)
            - produto (string)
            - preco_venda_unit (double)
            - qtd_produzida_mes (long)
            - custo_mat_unit (double)
            - peso_agua, peso_luz, peso_gas, peso_aluguel,
              peso_internet, peso_gasolina, peso_seguro (long)
            - horas_mao_obra_mes_produto (long)
            - mes_referencia (date)
            - data_execucao (timestamp)

    df_config:
        Schema esperado:
            - cliente (string)
            - mes_referencia (date)
            - luz, agua, aluguel, gas, internet, gasolina, seguro (long)
            - mao_de_obra_total_mes (long)
            - horas_mao_obra_mes (long)
            - data_execucao (timestamp)  # será descartada aqui para evitar duplicidade

    Retorno:
    --------
    DataFrame:
        Contém custos alocados, custos unitários, margens, markup e preços-alvo.
    """

    # -------------------------
    # 0) Remover data_execucao duplicada do df_config
    # -------------------------
    df_config_limpo = df_config.drop("data_execucao")

    # -------------------------
    # 1) Somar pesos por cliente + mês
    # -------------------------
    df_pesos_totais = (
        df_produtos
        .groupBy("cliente", "mes_referencia")
        .agg(
            F.sum("peso_agua").alias("peso_agua_total"),
            F.sum("peso_luz").alias("peso_luz_total"),
            F.sum("peso_gas").alias("peso_gas_total"),
            F.sum("peso_aluguel").alias("peso_aluguel_total"),
            F.sum("peso_internet").alias("peso_internet_total"),
            F.sum("peso_gasolina").alias("peso_gasolina_total"),
            F.sum("peso_seguro").alias("peso_seguro_total"),
        )
    )

    # -------------------------
    # 2) Base: produtos + config + pesos_totais
    # -------------------------
    df_base = (
        df_produtos.alias("p")
        .join(
            df_config_limpo.alias("c"),
            on=["cliente", "mes_referencia"],
            how="left"
        )
        .join(
            df_pesos_totais.alias("w"),
            on=["cliente", "mes_referencia"],
            how="left"
        )
    )

    # -------------------------
    # 3) Mão de obra (custo hora, custo do produto)
    # -------------------------
    df_calc = (
        df_base
        .withColumn(
            "custo_hora_mo",
            F.when(F.col("horas_mao_obra_mes") > 0,
                   F.col("mao_de_obra_total_mes") / F.col("horas_mao_obra_mes"))
             .otherwise(F.lit(0.0))
        )
        .withColumn(
            "custo_mo_prod_mes",
            F.col("custo_hora_mo") * F.col("horas_mao_obra_mes_produto")
        )
    )

    # -------------------------
    # 4) Função auxiliar para rateio de cada custo fixo
    # -------------------------
    def rateio(col_valor, col_peso_prod, col_peso_total):
        return F.when(
            F.col(col_peso_total) > 0,
            F.col(col_valor) * (F.col(col_peso_prod) / F.col(col_peso_total))
        ).otherwise(F.lit(0.0))

    df_calc = (
        df_calc
        # Água
        .withColumn(
            "custo_agua_prod_mes",
            rateio("agua", "peso_agua", "peso_agua_total")
        )
        # Luz
        .withColumn(
            "custo_luz_prod_mes",
            rateio("luz", "peso_luz", "peso_luz_total")
        )
        # Gás
        .withColumn(
            "custo_gas_prod_mes",
            rateio("gas", "peso_gas", "peso_gas_total")
        )
        # Aluguel
        .withColumn(
            "custo_aluguel_prod_mes",
            rateio("aluguel", "peso_aluguel", "peso_aluguel_total")
        )
        # Internet
        .withColumn(
            "custo_internet_prod_mes",
            rateio("internet", "peso_internet", "peso_internet_total")
        )
        # Gasolina
        .withColumn(
            "custo_gasolina_prod_mes",
            rateio("gasolina", "peso_gasolina", "peso_gasolina_total")
        )
        # Seguro
        .withColumn(
            "custo_seguro_prod_mes",
            rateio("seguro", "peso_seguro", "peso_seguro_total")
        )
    )

    # -------------------------
    # 5) Somar custos fixos do produto no mês
    # -------------------------
    df_calc = (
        df_calc
        .withColumn(
            "custo_fixos_total_mes",
            F.col("custo_agua_prod_mes") +
            F.col("custo_luz_prod_mes") +
            F.col("custo_gas_prod_mes") +
            F.col("custo_aluguel_prod_mes") +
            F.col("custo_internet_prod_mes") +
            F.col("custo_gasolina_prod_mes") +
            F.col("custo_seguro_prod_mes") +
            F.col("custo_mo_prod_mes")
        )
    )

    # -------------------------
    # 6) Custo fixo unitário + custo total unitário
    # -------------------------
    df_calc = (
        df_calc
        .withColumn(
            "custo_fixos_unit",
            F.when(
                F.col("qtd_produzida_mes") > 0,
                F.col("custo_fixos_total_mes") / F.col("qtd_produzida_mes")
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "custo_unit_total",
            F.col("custo_mat_unit") + F.col("custo_fixos_unit")
        )
    )

    # -------------------------
    # 7) KPIs de margem, markup, faturamento, preços-alvo
    # -------------------------
    df_silver = (
        df_calc
        # Faturamento e lucro
        .withColumn(
            "faturamento_mes",
            F.col("preco_venda_unit") * F.col("qtd_produzida_mes")
        )
        .withColumn(
            "margem_bruta_unit",
            F.col("preco_venda_unit") - F.col("custo_unit_total")
        )
        .withColumn(
            "margem_bruta_pct",
            F.when(
                F.col("preco_venda_unit") > 0,
                F.col("margem_bruta_unit") / F.col("preco_venda_unit")
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "lucro_total_mes",
            F.col("margem_bruta_unit") * F.col("qtd_produzida_mes")
        )
        .withColumn(
            "markup",
            F.when(
                F.col("custo_unit_total") > 0,
                F.col("preco_venda_unit") / F.col("custo_unit_total")
            ).otherwise(F.lit(None).cast("double"))
        )
        # Preços-alvo para margens desejadas
        .withColumn(
            "preco_target_30pct",
            F.when(
                F.col("custo_unit_total") > 0,
                F.col("custo_unit_total") / F.lit(1 - 0.30)
            ).otherwise(F.lit(None).cast("double"))
        )
        .withColumn(
            "preco_target_40pct",
            F.when(
                F.col("custo_unit_total") > 0,
                F.col("custo_unit_total") / F.lit(1 - 0.40)
            ).otherwise(F.lit(None).cast("double"))
        )
        .withColumn(
            "preco_target_50pct",
            F.when(
                F.col("custo_unit_total") > 0,
                F.col("custo_unit_total") / F.lit(1 - 0.50)
            ).otherwise(F.lit(None).cast("double"))
        )
    )

    return df_silver


In [0]:


def merge_silver(df_silver: DataFrame, tabela_destino: str) -> None:
    """
    Faz MERGE da Silver na tabela Delta de destino, logando o que está acontecendo.
    Chave de merge: (cliente, produto, mes_referencia)
    """

    # Contagem dos registros de entrada
    print("🔧 Iniciando MERGE da Silver...")
    print(f"🎯 Tabela de destino: {tabela_destino}")

    if spark.catalog.tableExists(tabela_destino):
        print(f"📦 Tabela {tabela_destino} já existe — realizando MERGE...")

        delta = DeltaTable.forName(spark, tabela_destino)

        # Executa o MERGE
        (
            delta.alias("t")
            .merge(
                df_silver.alias("s"),
                """
                t.cliente = s.cliente
                AND t.produto = s.produto
                AND t.mes_referencia = s.mes_referencia
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"✅ MERGE concluído com sucesso!")

    else:
        print(f"🆕 Tabela {tabela_destino} NÃO existe — criando pela primeira vez...")
        df_silver.write \
                .format("delta") \
                .partitionBy("mes_referencia") \
                .mode("append") \
                .option("overwriteSchema", "true") \
                .saveAsTable(tabela_destino)
        print(f"✅ Tabela {tabela_destino} criada com sucesso!!!")


In [0]:
def gerar_gold_contabilidade(df_silver: DataFrame):
    """
    Gera 2 visões GOLD a partir da Silver de contabilidade de produtos:

    1) gold_produto -> visão por produto, custos, margens e faturamento
    2) gold_cliente -> visão consolidada por cliente, incluindo
                       qual produto mais prejudicou o cliente no mês
    """

    # ---------------------------------------------------------------------
    # GOLD 1 — Visão por produto (cliente + produto + mês)
    # ---------------------------------------------------------------------
    gold_produto = (
        df_silver
        .select(
            "cliente",
            "produto",
            "mes_referencia",

            # Preço e custos unitários
            F.col("custo_unit_total").alias("custo_total_unitario"),
            F.col("custo_mat_unit").alias("custo_materia_prima_unitaria"),
            F.col("custo_fixos_unit").alias("custo_fixo_unitario"),

            # Custos mensais por produto
            F.col("custo_mo_prod_mes").alias("custo_mao_de_obra_mensal_por_produto"),
            F.col("custo_fixos_total_mes").alias("custo_fixo_total_mensal"),

            # Preço de venda
            F.col("preco_venda_unit").alias("preco_venda_unitario"),

            # Margens
            F.col("margem_bruta_unit").alias("margem_bruta_unitaria"),
            F.col("margem_bruta_pct").alias("margem_bruta_percentual"),

            # Resultado mensal
            F.col("faturamento_mes").alias("faturamento_total_mensal"),
            F.col("lucro_total_mes").alias("lucro_bruto_total_mensal"),

            # Targets de margem
            F.col("preco_target_30pct").alias("preco_minimo_para_margem_30pct"),
            F.col("preco_target_40pct").alias("preco_minimo_para_margem_40pct"),
            F.col("preco_target_50pct").alias("preco_minimo_para_margem_50pct"),

            # Quantidade produzida
            "qtd_produzida_mes"
        )
    )

    # ---------------------------------------------------------------------
    # GOLD 2 — Visão consolidada por cliente + mês
    # ---------------------------------------------------------------------

    # 2.1 Consolidação geral
    gold_cliente_base = (
        df_silver
        .groupBy("cliente", "mes_referencia")
        .agg(
            F.sum("faturamento_mes").alias("faturamento_total_mensal"),
            F.sum("lucro_total_mes").alias("lucro_bruto_total_mensal"),

            F.avg("margem_bruta_pct").alias("margem_bruta_percentual_media"),
            F.avg("margem_bruta_unit").alias("margem_bruta_unitaria_media"),

            F.sum("custo_fixos_total_mes").alias("custo_fixo_total_mensal"),
            F.sum("custo_mo_prod_mes").alias("custo_mao_de_obra_total_mensal"),

            F.sum("qtd_produzida_mes").alias("quantidade_total_produzida"),

            (
                F.sum(F.when(F.col("margem_bruta_unit") < 0, 1).otherwise(0))
                / F.count("*")
            ).alias("percentual_produtos_com_prejuizo")
        )
        .withColumn(
            "ticket_medio_unitario",
            F.col("faturamento_total_mensal") / F.col("quantidade_total_produzida")
        )
    )

    # 2.2 Produto que mais prejudica o cliente
    janela_pior_produto = (
        Window
        .partitionBy("cliente", "mes_referencia")
        .orderBy(F.col("lucro_total_mes").asc())  # mais negativo primeiro
    )

    df_pior_produto = (
        df_silver
        .withColumn("rn", F.row_number().over(janela_pior_produto))
        .filter(F.col("rn") == 1)
        .select(
            "cliente",
            "mes_referencia",
            F.col("produto").alias("produto_mais_prejuizo"),
            F.col("lucro_total_mes").alias("lucro_bruto_produto_mais_prejuizo"),
            F.col("margem_bruta_pct").alias("margem_produto_mais_prejuizo")
        )
    )

    # 2.3 Junta tudo no GOLD
    gold_cliente = (
        gold_cliente_base
        .join(
            df_pior_produto,
            on=["cliente", "mes_referencia"],
            how="left"
        )
    )

    # ---------------------------------------------------------------------
    # Retorna apenas as 2 tabelas GOLD
    # ---------------------------------------------------------------------
    return gold_produto, gold_cliente

In [0]:
def gold_produtos_maior_prejuizo(df_silver):

    df = (
        df_silver
        .select(
            "cliente",
            "produto",
            "mes_referencia",
            F.col("lucro_total_mes"),
            F.col("margem_bruta_unit").alias("margem_bruta_unitaria"),
            F.col("custo_unit_total").alias("custo_total_unitario"),
            F.col("preco_venda_unit").alias("preco_venda_unitario"),
            F.col("preco_target_30pct").alias("preco_minimo_para_margem_30pct"),
        )
        # Apenas produtos prejudiciais
        .where(F.col("lucro_total_mes") < 0)
        # Quanto precisa reajustar
        .withColumn(
            "sugestao_reajuste_percentual",
            (F.col("preco_minimo_para_margem_30pct") / F.col("preco_venda_unitario")) - 1
        )
        # Ranking: 1 = maior prejuízo do cliente no mês
        .withColumn(
            "ranking_prejuizo",
            F.row_number().over(
                Window.partitionBy("cliente", "mes_referencia")
                      .orderBy(F.col("lucro_total_mes").asc())  # mais negativo primeiro
            )
        )
    )

    return df


In [0]:
def merge_delta_gold(df: DataFrame, tabela_destino: str) -> None:
    """
    Realiza MERGE (upsert) de um DataFrame em uma tabela Delta Lake da camada GOLD
    de contabilidade por produto.

    Contexto:
    ---------
    - Tabela de fato mensal de resultado por produto.
    - Grão: 1 linha por (cliente, produto, mes_referencia).

    Regras de chave:
    ----------------
    - Chave de negócio: (cliente, produto, mes_referencia)
    - Mantém sempre APENAS 1 linha por (cliente, produto, mes_referencia)
      na tabela destino.
    - Caso o DataFrame de entrada contenha mais de um registro para o mesmo
      (cliente, produto, mes_referencia), será mantida apenas 1 linha de forma
      arbitrária (primeira encontrada na janela). Se quiser outra regra
      (ex: maior faturamento), é só ajustar o critério de ordenação da janela.

    Parâmetros:
    -----------
    df : DataFrame
        DataFrame com os dados GOLD a serem inseridos/atualizados.
        Deve conter, obrigatoriamente:
            - cliente
            - produto
            - mes_referencia

    tabela_destino : str
        Nome da tabela Delta Lake, por exemplo:
            - "workspace.gold_contabilidade.margens_por_produto"

    Retorno:
    --------
    None
    """

    # ⚠️ Verifica se há dados
    if df is None or df.limit(1).count() == 0:
        print("⚠️ Nenhum dado para atualizar na GOLD.")
        return

    # 🔁 Deduplica por (cliente, produto, mes_referencia)
    # Mantém apenas 1 linha por chave para evitar erro de MERGE com múltiplos matches.
    janela = (
        Window
        .partitionBy("cliente", "produto", "mes_referencia")
        .orderBy(F.lit(1))  # critério "qualquer"; ajuste se quiser uma regra específica
    )

    df_dedup = (
        df
        .withColumn("_rn", F.row_number().over(janela))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    total = df_dedup.count()
    print(f"✅ Total de registros GOLD (deduplicados por cliente/produto/mês) a carregar: {total}")
    print(f"🎯 Tabela de destino GOLD: {tabela_destino}")

    # 🔍 Verifica se a tabela Delta já existe
    if spark.catalog.tableExists(tabela_destino):
        print(f"📦 Tabela {tabela_destino} já existe — realizando MERGE...")

        delta_table = DeltaTable.forName(spark, tabela_destino)

        # 🔑 Condição de MERGE pela chave de negócio GOLD
        condicao_merge = """
            t.cliente        = s.cliente
        AND t.produto        = s.produto
        AND t.mes_referencia = s.mes_referencia
        """
        print("🔑 Usando chaves: cliente, produto, mes_referencia")

        (
            delta_table.alias("t")
            .merge(
                df_dedup.alias("s"),
                condicao_merge
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"✅ MERGE GOLD concluído com sucesso em {tabela_destino}")

    else:
        print(f"🆕 Tabela {tabela_destino} não existe — criando nova tabela GOLD...")

        (
            df_dedup.write
            .format("delta")
            .partitionBy("mes_referencia")  # particionar por mês faz bastante sentido aqui
            .mode("append")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_destino)
        )

        print(f"✅ Tabela GOLD {tabela_destino} criada e dados inseridos com sucesso.")
